# ಪಾಠ 17 - Foundry Local ಮತ್ತು Qwen ಜೊತೆಗೆ ಸ್ಥಳೀಯ AI ಏಜೆಂಟ್ಸ್ ರಚನೆ

ಈ ನೊಟ್‌ಬುಕ್‌ನಲ್ಲಿ ನೀವು ಸಂಪೂರ್ಣವಾಗಿ ನಿಮ್ಮ ವರ್ಕ್‌ಸ್ಟೇಶನ್‌ನಲ್ಲಿ ನಡೆಯುವ **ಸ್ಥಳೀಯ ಎಂಜಿನಿಯರಿಂಗ್ ಸಹಾಯಕ** ಅನ್ನು ನಿರ್ಮಿಸುತ್ತೀರಿ. ಯಾವುದೇ ಸಮಯದಲ್ಲೂ ಕ್ಲೌಡ್ ಇನ್ಫರೆನ್ಸ್ ಬಳಸಲಾಗುವುದಿಲ್ಲ. ಸಹಾಯಕವು ಮಾಡಬಹುದಾದವುಗಳು:

1. Foundry Local ಮೂಲಕ Qwen ಫಂಕ್ಷನ್ ಕಾಲಿಂಗ್ ಮೂಲಕ **ಉಪಕರಣಗಳನ್ನು ಕರೆ ಮಾಡಲು**.
2. ಸ್ಯಾಂಡ್ಬಾಕ್ಸ್ ಯೋಜನಾ ಡೈರೆಕ್ಟರಿಯೊಳಗಿನ **ಫೈಲ್‌ಗಳನ್ನು ಪಟ್ಟಿ ಮಾಡುತೆ ಮತ್ತು ಓದಿ**.
3. ಸರಳ ಮ್ಯಾಟ್ರಿಕ್ಸ್‌ಗಳೊಂದಿಗೆ **ಕೋಡ್ ವಿಶ್ಲೇಷಣೆ ಮಾಡಲು**.
4. ಸ್ಥಳೀಯ RAG (Chroma) ಬಳಸಿ **ದಾಖಲಾತಿಗಳ ಹುಡುಕಾಟ**.
5. **ಸ್ಥಳೀಯ MCP сервер ಬಳಸಿ** (ಯಾವುದೇ ಸಂರಚನೆಯಿಲ್ಲದಿದ್ದರೆ graceful ಆಗಿ ಬಿಟ್ಟುಬಿಡಲಾಗುತ್ತದೆ).

ಏಜೆಂಟ್ ಕೋಡ್ ಕ್ಲೌಡ್ ಪಾಠಗಳಿಗೆ ಬಹುಮಾನವಾಗಿ ಹೋಲಿಕೆಯಂತೆ ಕಾಣುತ್ತದೆ — ಕೇವಲ 클라우ಡ್ ಕ್ಲೈಯೆಂಟ್ ಎಂಡ್ಪಾಯಿಂಟ್ `localhost` ಗೆ ಸ್ಥಳಾಂತರಿತವಾಗಿದೆ.


## ಸಿದ್ಧತೆ

ಈ ನೋಟ್ಬುಕ್ ಅನ್ನು ಚಾಲನೆ ಮಾಡುವ ಮೊದಲು:

1. **Microsoft Foundry Local ಅನ್ನು ಸ್ಥಾಪಿಸಿ** (ನಿಮ್ಮ OS ಗಾಗಿ [ಡಾಕ್ಯುಮೆಂಟೇಷನ್](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) ನೋಡಿ).
2. **Qwen ಮಾದರಿಯನ್ನು ಡೌನ್‌ಲೋಡ್ ಮಾಡಿ ಮತ್ತು ಪ್ರಾರಂಭಿಸಿ:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. ಕೆಳಗಿನ Python ಪ್ಯಾಕೇಜ್‌ಗಳನ್ನು ಸ್ಥಾಪಿಸಿ.

ಎಲ್ಲವೂ ಸ್ಥಳೀಯವಾಗಿ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ. ~8 GB RAM ಇರುವ ಯಂತ್ರವು reಅರ್ಜಿ ಕನಿಷ್ಟವಾಗಿದೆ.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. ಫೌಂಡ್ರಿ ಲೋಕಲ್ ಗೆ ಸಂಪರ್ಕಿಸು

`FoundryLocalManager` ಮಾದರಿಯನ್ನು ಅಗತ್ಯವಿದ್ದರೆ ಡೌನ್‌ಲೋಡ್ ಮಾಡುತ್ತದೆ, ಲೋಕಲ್ ಸೇವೆಯನ್ನು ಪ್ರಾರಂಭಿಸುತ್ತದೆ, ಮತ್ತು ನಮಗೆ **OpenAI-ಅನುಕೂಲ ಎಂಡ್ಪಾಯಿಂಟ್** ಒದಗಿಸುತ್ತದೆ. ನಂತರ ನಾವು ಸಾಮಾನ್ಯ OpenAI SDK ಅನ್ನು ಅದಕ್ಕೆ ಸೂಚಿಸುತ್ತೇವೆ. API ಕೀ ಒಂದು ಸ್ಥಳೀಯ ಪ್ಲೇಸ್‌ಹೋಲ್ಡರ್ — ಯಾವುದೇ ಮೇಘ ಪ್ರಮಾಣಪತ್ರ涉ರಿಸುವುದಿಲ್ಲ.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. ಸ್ಥಳೀಯ ಸಾಧನಗಳು (Sandboxed ಫೈಲ್ ಕಾರ್ಯಾಚರಣೆಗಳು)

ನಾವು ಡಿಸ್ಕ್‌ನಲ್ಲಿ ಒಂದು ಸಣ್ಣ ಮಾದರಿ ಪ್ರಾಜೆಕ್ಟ್ ರಚಿಸುತ್ತೇವೆ, ನಂತರ ಆ ಪ್ರಾಜೆಕ್ಟ್ ರೂಟ್‌ಗೆ ಸೀಮಿತವಾಗಿರುವ ಸಾಧನಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತೇವೆ. ಸ್ಯಾಂಡ್‌ಬಾಕ್ಸ್ ಪರಿಶೀಲನೆ ಸ್ಥಳೀಯವಾಗಿಯೂ ಮುಖ್ಯವಾಗಿದೆ: ಯಾವುದೇ ಪಥಗಳನ್ನು ಓದುವ ಸಾಧನವು ನಿಮ್ಮ ಬಳಕೆದಾರರ ಅನುಮತಿಗಳೊಂದಿಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ ಮತ್ತು ನೀವು ಸ್ಪರ್ಶಿಸಬಹುದಾದ ಎಲ್ಲವನ್ನು ಸ್ಪರ್ಶಿಸಬಹುದು.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. ಕ್ಲೋಚಿತ್ರೊೊಂದಿಗೆ ಸ್ಥಳೀಯ RAG

ನಾವು ಕೆಲವು ಡಾಕ್ಯುಮೆಂಟೇಷನ್ ತುಣುಕುಗಳನ್ನು ಸ್ಥಳೀಯ ಕ್ಲೋಚಿತ್ರೊ ಸಂಗ್ರಹಣೆಗೆಕ್ಕೆ ಸೇರಿಸುತ್ತೇವೆ. ಕ್ಲೋಚಿತ್ರೊ ಪ್ರಕ್ರಿಯೆಯೊಳಗೆ ನಿರ್ವಹಿಸುವ ಮೂಲಕ ವೆಕ್ಟರ್‌ಗಳನ್ನು ಡಿಸ್ಕ್‌ನಲ್ಲಿ ಸಂರಕ್ಷಿಸುತ್ತದೆ — ಯಾವುದೇ ಸರ್ವರ್ ಅಥವಾ ಕ್ಲೌಡ್ ಇಲ್ಲ. `search_docs` ಉಪಕರಣವು ಪ್ರಶ್ನೆಗೆ ಸಂಬಂಧಿಸಿದ ಅತ್ಯಂತ ಪ್ರಸ್ತುತ ತುಣುಕುಗಳನ್ನು ಪಡೆಯುತ್ತದೆ.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. ಸಾಧನ-ಕರೆದ ಲೂಪ್  

ಈಗ ನಾವು OpenAI ಸಾಧನ ಸ್ಕೀಮಾ ಬಳಸಿ ಸಾಧನಗಳನ್ನು ಮಾದರಿಯೊಂದಿಗೆ ನೋಂದಾಯಿಸಿ, ಸ್ಟ್ಯಾಂಡರ್ಡ್ ಸಾಧನ-ಕರೆಗೆಲೂಪ್ ಅನ್ನು ನಡೆಸುತ್ತೇವೆ — ಮಾದರಿ ಸಾಧನವನ್ನು ಕೇಳಿಸುತ್ತದೆ, ನಾವು ಅದನ್ನು ಸ್ಥಳೀಯವಾಗಿ ಕಾರ್ಯಗತಗೊಳಿಸಿ ಫಲಿತಾಂಶವನ್ನು ಹಿಂತಿರುಗಿಸುತಿ, ಮತ್ತು ಪ್ರತಿಕ್ರಿಯಿಸುವವರೆಗೆ ಈ ಕ್ರಮವನ್ನು繰り返ಿಸುತ್ತೇವೆ. Qwen ನ ವಿಶ್ವಸನೀಯ ಫಂಕ್ಷನ್ ಕರೆಗೆನವುವು ಈ ಕಾರ್ಯವನ್ನು ಸಾಧನದ ಒಳಗೆ ಸಾಧ್ಯವಾಗಿಸುತ್ತದೆ.  


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. ಸ್ಥಳೀಯ MCP (ಐಚ್ಛಿಕ)

MCP ಒಂದು ಟ್ರಾನ್ಸ್‌ಪೋರ್ಟ್ ಆಗಿದ್ದು, ಕ್ಲೌಡ್ ಸೇವೆ ಅಲ್ಲ — MCP ಸರ್ವರ್ ಅನ್ನು `stdio`ಮೇಲೆ ಸ್ಥಳೀಯ ಪ್ರಕ್ರಿಯೆಯಾಗಿ ರೂಪಿಸಲಾಗಬಹುದು. ಕೆಳಗಿನ ಕೋಷವು ನಿಮ್ಮ ಬಳಿ ಕಾನ್ಫಿಗರ್ ಮಾಡಲಾದ ಸ್ಥಳೀಯ MCP ಸರ್ವರ್‌ನೊಂದಿಗೆ (ಉದಾಹರಣೆಗೆ, ಫೈಲ್‌ಸಿಸ್ಟಮ್ ಸರ್ವರ್) ಸಂಪರ್ಕ ಹೇಗೆ ಮಾಡಬೇಕೆಂಬುದನ್ನು ತೋರಿಸುತ್ತದೆ. `LOCAL_MCP_COMMAND` ಸೆಟ್ ಆಗಿರದಿದ್ದರೆ ಅದು ಸೌಮ್ಯದಾಗಿ ಬಿಟ್ಟುಹೋಗುತ್ತದೆ, ಆದ್ದರಿಂದ ನೋಟ್‌ಬುಕ್ ಅಂತಿಮವಾಗಿ ಕಾಣಿಸಿಕೊಳ್ಳುತ್ತದೆ.

ಭದ್ರತಾ ಸೂಚನೆ: ಸ್ಥಳೀಯ MCP ಸರ್ವರ್ ನಿಮ್ಮ ಬಳಕೆದಾರದ ಅನುಮತಿಗಳೊಂದಿಗೆ ಕಾರ್ಯನಿರ್ವಹಿಸುತ್ತದೆ. ಅದನ್ನು ಒಂದು ಯೋಜನೆ ಡೈರಕ್ಟರ್‌ಗೆ ಪರಿಧಿ ಮಾಡುವಿರಿ ಮತ್ತು ಅದರ ಔಟ್‌ಪುಟ್‌ಗಳನ್ನು ಪರಿಶೀಲಿಸಿ ನಂತರ ಅದನ್ನು ಬಳಸಿರಿ.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## ಸಾರಾಂಶ

ನೀವು ಸಂಪೂರ್ಣವಾಗಿ ನಿಮ್ಮ ಯಂತ್ರದಲ್ಲಿ ನಡೆಯುವ ಇಂಜಿನಿಯರಿಂಗ್ ಸಹಾಯಕನನ್ನು ನಿರ್ಮಿಸಿದ್ದೀರಿ:

- **ಫೌಂಡ್ರಿ ಸ್ಥಳೀಯ** OpenAI-ಸಹಕಾರ್ಯದಂತ ಅಂತಿಮ ಬಿಂದು ಹಿಂದೆ **Qwen** ಮಾದರಿಯನ್ನು ಸರ್ವ್ ಮಾಡಿತು — ಆದ್ದರಿಂದ ಏಜೆಂಟ್ ಕೋಡ್ ಕ್ಲೌಡ್ ಪಾಠಗಳನ್ನು ಹೊಂದಿದೆ.
- **ಸ್ಯಾಂಡ್ಬಾಕ್ಸ್ ಉಪಕರಣಗಳು** ಏಜೆಂಟ್ ಗೆ ಪ್ರಾಜೆಕ್ಟ್ ಡೈರೆಕ್ಟರಿಯನ್ನು ತಲುಪದೆ ಫೈಲ್ ಪ್ರವೇಶ ಮತ್ತು ಕೋಡ್ ವಿಶ್ಲೇಷಣೆ ನೀಡಿತು.
- **ಕ್ರೋಮಾ** ಪ್ರಾರಂಭಿಕ **ಸ್ಥಳೀಯ RAG** ಅನ್ನು ಡಾಕ್ಯುಮೆಂಟೇಷನ್ ಮೇಲೆ ಒದಗಿಸಿತು.
- **ಸ್ಥಳೀಯ MCP** ಆಫ್‌ಲೈನ್‌ನಲ್ಲಿ MCP ಪರಿಸರವನ್ನು ಮರುಬಳಕೆ ಮಾಡುವುದನ್ನು ತೋರಿಸಿತು.

ಯಾವಾಗಲೂ ಯಾವುದೇ ಕ್ಲೌಡ್ ನಿರ್ಣಯವನ್ನು ಬಳಸಲಾಗಲಿಲ್ಲ.

### ಸವಾಲು

ಸ್ಥಳೀಯ ಏಜೆಂಟ್ ಅನ್ನು ವಿಸ್ತರಿಸಿ:

1. **ಬಹು MCP ಸರ್ವರ್‌ಗಳೊಂದಿಗೆ ಕೆಲಸ ಮಾಡು** — ಫೈಲ್ ಸಿಸ್ಟಮ್ ಸರ್ವರ್ ಮತ್ತು ಗಿಟ್ ಸರ್ವರ್ ಅನ್ನು ಸಂಪರ್ಕಿಸಿ ಮತ್ತು ಏಜೆಂಟ್ ಅವುಗಳ ನಡುವೆ ಆಯ್ಕೆಮಾಡಿಕೊಳ್ಳಲು ಬಿಡಿ.
2. **ಸ್ಥಳೀಯ ಮೆಮೊರಿ ಬಳಸಿ** — مختصر ಸಂಭಾಷಣೆಯ ಇತಿಹಾಸವನ್ನು ಡಿಸ್‌ಕ್‌ಗೆ ಉಳಿಸಿ ಹಾಗಾಗಿ ಸಹಾಯಕನು ನೋಟ್ಬುಕ್ ಮರುಪ್ರಾರಂಭಗಳ ನಡುವೆ ಹಳೆಯ ವಾಗ್ದಾನಗಳನ್ನು ನೆನಪಿಡುತ್ತಾನೆ.
3. **ಸ್ಥಳೀಯ ಬಹು-ಏಜೆಂಟ್ ಏರ್ಪಾಡು ಬೆಂಬಲಿಸಿ** — ಎರಡನೇ ಸ್ಥಳೀಯ ಏಜೆಂಟ್ (ಉದಾ: ವಿಮರ್ಶಕ) ಸೇರಿಸಿ ಮತ್ತು ಇಬ್ಬರೂ ಟಾಸ್ಕ್‌ನಲ್ಲಿ ಸಹಕಾರ ಮಾಡಲಿ.

ಮುಂದಿನ ಪಾಠದಲ್ಲಿ ನೀವು ನಿಯೋಜಿಸಲಾದ AI ಏಜೆಂಟ್‌ಗಳನ್ನು ಸುರಕ್ಷಿತಗೊಳಿಸುವುದನ್ನು ಕಲಿತೀರಿ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
